In [1]:
### Load the json data ###

import json

def dict_load(json_path):
    """
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

In [2]:
### Get the keys hierarchy for the json in a file ###

def keyy(data):
    """
    """
    seen_keys = set()
    res = set()

    def walk(obj, prefix="/"):

        if isinstance(obj, dict):
            for k, v in obj.items():
                if k not in seen_keys:
                    res.add(prefix + k)
                    seen_keys.add(k)
                    # print(f"{prefix}- {k}")
                walk(v, prefix + k + "/")

        elif isinstance(obj, list):
            for item in obj:
                walk(item, prefix)

    walk(data)

    return res

def keyy_ff(json_path):
    """
    """
    data = dict_load(json_path)
    res = keyy(data)

    return res

import json
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

def binary_metrics(y_true, y_pred, pos_label='Causal'):
    """
    Helper to calculate micro precision, recall, and F1 for a specific positive label.
    """
    # We use average='binary' here because we have mapped everything to 
    # a binary problem (Causal vs NoRel).
    # If the set is empty (e.g., no predictions for a specific type), handle gracefully.
    try:
        p, r, f, _ = precision_recall_fscore_support(
            y_true, y_pred, pos_label=pos_label, average='binary', zero_division=0
        )
    except ValueError:
        return 0.0, 0.0, 0.0
        
    return {
        "precision": round(p, 4),
        "recall": round(r, 4),
        "f1": round(f, 4)
    }

In [3]:
def evaluate_whole(df, pred_name: str="unified_pred", gold_name: str="unified_gold"):
    """
    """
    res = {}
    
    res = binary_metrics(
        df[gold_name], 
        df[pred_name]
    )
    
    return res

In [4]:
def evaluate_per_lang(df, pred_name: str="unified_pred", gold_name: str="unified_gold"):
    """
    Per Language Unified Scores
    """
    res = {}

    # Lising the language in the data
    languages = df['lang'].unique()
    
    for lang in languages:
        lang_df = df[df['lang'] == lang]
        res[lang] = binary_metrics(
            lang_df[gold_name], 
            lang_df[pred_name]
        )
    return res

In [5]:
def reverse_pair_key(pair_str):
    """
    Flips 'T10,T11' to 'T11,T10'.
    """
    parts = pair_str.split(',')
    if len(parts) != 2: raise ValueError(f"{pair_str} is not a reversible pair")
    return f"{parts[1]},{parts[0]}"

In [6]:
def flatten_pairs_solo(df, key_name: str="gold"):
    """
    """
    res = set()
    
    for _, row in df.iterrows():
        res.add((row['id'], row['pair'], row[key_name]))
    
    return res

In [7]:
def causal_simple(df_part, causal_list=["EffectCause", "CauseEffect"]):
    """
    """
    res = df_part.apply(lambda x: 'Causal' if x in causal_list else 'NoRel')
    return res

In [8]:
# NoClip not implemented
def symmetrize(df, rel_name="CauseEffect", key_name="gold", no_clip=False, default_rel='NoRel'):
    """
    """
    flat = flatten_pairs_solo(df, key_name=key_name)
    # print(flat)
    flat_exist = set(line[:2] for line in flat)
    # print("flat[0]", flat_exist[0])
    
    def symmetrize_row(row):
        rev_pair = reverse_pair_key(row['pair'])
        if not (row['id'], rev_pair) in flat_exist and no_clip:
            # if row['id']=="conflict-week4-isik-2981956_chunk_4.ann":
            #     print(row['id'], row['pair'], (row['id'], rev_pair) in flat_exist)
            return default_rel
        if row[key_name] == rel_name:
            return rel_name
        if (row['id'], rev_pair, rel_name) in flat:
            # print(row['id'], rev_pair, rel_name)
            # print("uwu")
            return rel_name
        return default_rel
    
    return df.apply(symmetrize_row, axis=1)

In [9]:
def evaluate(df, pred_name: str="unified_pred", gold_name: str="unified_gold"):
    """
    """
    res = {}
    res['per_lang'] = evaluate_per_lang(df, pred_name=pred_name, gold_name=gold_name)
    res['overall'] = evaluate_whole(df, pred_name=pred_name, gold_name=gold_name)
    return res

In [10]:
def data_to_df(data):
    """
    """
    predictions = data['results']['per_pair_predictions']
    # Dataframe
    df = pd.DataFrame(predictions)
    return df

In [36]:
def analyse(data):
    # Get individual predictions from the file
    res = {}
    gold_name = 'unified_gold'
    binary_pred = 'binary_pred'
    all_rel = ['CauseEffect', 'EffectCause']
    ec_rel = ['EffectCause']
    ce_rel = ['CauseEffect']
    
    predictions = data['results']['per_pair_predictions']
    # Dataframe
    df = pd.DataFrame(predictions)
    # Unify the gold
    df[gold_name] = causal_simple(df['gold'], all_rel)
    # Binarize the preds 
    df[binary_pred] = causal_simple(df['pred'], all_rel)
    # Binarize the preds from EffectCause only
    df["ec_only"] = causal_simple(df['pred'], ec_rel)
    df["ce_only"] = causal_simple(df['pred'], ce_rel)
    
    df["clipped_sym_ec"] = symmetrize(df, rel_name="Causal", key_name="ec_only", no_clip=True, default_rel='NoRel')
    df["sym_ec"] = symmetrize(df, rel_name="Causal", key_name="ec_only", no_clip=False, default_rel='NoRel')
    df["sym_ce"] = symmetrize(df, rel_name="Causal", key_name="ce_only", no_clip=False, default_rel='NoRel')

    # return df[:50]
    
    res["direct_binary"] = evaluate(df, pred_name=binary_pred, gold_name=gold_name)
    res["clipped_sym_ec"] = evaluate(df, pred_name="clipped_sym_ec", gold_name=gold_name)
    res["sym_ec"] = evaluate(df, pred_name="sym_ec", gold_name=gold_name)
    res["sym_ce"] = evaluate(df, pred_name="sym_ce", gold_name=gold_name)
    
    return res

In [12]:
def analyse_ff(json_path):
    """
    """
    data = dict_load(json_path)
    res = analyse(data)

    return res

In [38]:
keyy_ff("../agentere/logs/run_2026-02-05T15-33-45.728643p00-00_8e9ee038.json")

{'/cli_args',
 '/config',
 '/config/active_dataset',
 '/config/datasets',
 '/config/datasets/event_story_line',
 '/config/datasets/maven_ere',
 '/config/datasets/maven_ere/ann_field',
 '/config/datasets/maven_ere/labels',
 '/config/datasets/maven_ere/max_examples',
 '/config/datasets/maven_ere/name',
 '/config/datasets/maven_ere/prompt',
 '/config/datasets/maven_ere/repo_id',
 '/config/datasets/maven_ere/rule_set',
 '/config/datasets/maven_ere/split',
 '/config/datasets/maven_ere/text_field',
 '/config/datasets/meci',
 '/config/experiment',
 '/config/experiment/concurrency',
 '/config/experiment/enable_tools',
 '/config/experiment/resampling',
 '/config/experiment/resampling/enabled',
 '/config/experiment/resampling/n_runs',
 '/config/experiment/resampling/tie_breaking',
 '/config/experiment/retries',
 '/config/experiment/tools',
 '/config/experiment/tracing',
 '/config/experiment/tracing_name',
 '/config/model',
 '/config/model/base_url',
 '/config/model/default_model_id',
 '/config/m

In [5]:
keyy_ff("../aya/logs/run_2026-02-04T19-10-09.630784p00-00_1af359cb.json")

{'/cli_args',
 '/cli_args/ann_field',
 '/cli_args/concurrency',
 '/cli_args/logdir',
 '/cli_args/ls_project',
 '/cli_args/max_examples',
 '/cli_args/n_runs',
 '/cli_args/pair_batch_size',
 '/cli_args/repo',
 '/cli_args/resampling',
 '/cli_args/split',
 '/cli_args/streaming',
 '/cli_args/text_field',
 '/cli_args/tie_breaking',
 '/config',
 '/config/inference',
 '/config/inference/initial_prediction_mode',
 '/config/langsmith_project',
 '/config/model',
 '/config/openrouter_base_url',
 '/config/paths',
 '/config/paths/dataset_path',
 '/config/prompts',
 '/config/prompts/meci',
 '/config/prompts/meci/addon',
 '/config/prompts/meci/example_json',
 '/config/prompts/meci/user_template',
 '/config/prompts/roles',
 '/config/prompts/roles/critic',
 '/config/prompts/roles/summarizer',
 '/config/prompts/roles/thinker',
 '/config/prompts/roles/worker',
 '/config/prompts/system',
 '/config/prompts/system/base',
 '/config/resampling/aggregation',
 '/config/resampling/max_concurrency_per_run',
 '/con

In [39]:
df = data_to_df(dict_load("../agentere/logs/run_2026-02-05T15-33-45.728643p00-00_8e9ee038.json"))

In [108]:
a = df[:50].copy()
a["sym"] = symmetrize(a)
a

,doc_idx,id,lang,pair,gold,pred,vote_counts,sym
0,11,conflict-week4-isik-2981956_chunk_4.ann,causal-tr,"T10,T11",NoRel,NoRel,{'NoRel': 1},NoRel
1,11,conflict-week4-isik-2981956_chunk_4.ann,causal-tr,"T6,T7",CauseEffect,NoRel,{'NoRel': 1},CauseEffect
2,11,conflict-week4-isik-2981956_chunk_4.ann,causal-tr,"T2,T3",NoRel,NoRel,{'NoRel': 1},NoRel
3,11,conflict-week4-isik-2981956_chunk_4.ann,causal-tr,"T0,T3",NoRel,NoRel,{'NoRel': 1},NoRel
4,11,conflict-week4-isik-2981956_chunk_4.ann,causal-tr,"T8,T9",NoRel,NoRel,{'NoRel': 1},NoRel
5,11,conflict-week4-isik-2981956_chunk_4.ann,causal-tr,"T5,T0",CauseEffect,NoRel,{'NoRel': 1},CauseEffect
6,11,conflict-week4-isik-2981956_chunk_4.ann,causal-tr,"T2,T6",NoRel,NoRel,{'NoRel': 1},NoRel
7,11,conflict-week4-isik-2981956_chunk_4.ann,causal-tr,"T7,T6",EffectCause,NoRel,{'NoRel': 1},CauseEffect
8,11,conflict-week4-isik-2981956_chunk_4.ann,causal-tr,"T5,T7",NoRel,NoRel,{'NoRel': 1},NoRel
9,11,conflict-week4-isik-2981956_chunk_4.ann,causal-tr,"T11,T3",NoRel,NoRel,{'NoRel': 1},NoRel


In [32]:
from pprint import pprint
pprint(analyse_ff("../agentere/logs/run_2026-02-05T15-33-45.728643p00-00_8e9ee038.json")) # Coherence check 

{'clipped_sym_ec': {'overall': {'f1': 0.6775,
                                'precision': 0.7009,
                                'recall': 0.6557},
                    'per_lang': {'causal-da': {'f1': 0.6726,
                                               'precision': 0.6109,
                                               'recall': 0.748},
                                 'causal-en': {'f1': 0.7125,
                                               'precision': 0.8507,
                                               'recall': 0.6129},
                                 'causal-es': {'f1': 0.6088,
                                               'precision': 0.5035,
                                               'recall': 0.7698},
                                 'causal-tr': {'f1': 0.7002,
                                               'precision': 0.8228,
                                               'recall': 0.6093},
                                 'causal-ur': {'f1': 0.6488,
          

In [33]:
from pprint import pprint
pprint(analyse_ff("../agentere/logs/run_2026-02-05T10-36-03.414779p00-00_083c936b.json")) # No tool

{'clipped_sym_ec': {'overall': {'f1': 0.6348,
                                'precision': 0.7502,
                                'recall': 0.5502},
                    'per_lang': {'causal-da': {'f1': 0.6,
                                               'precision': 0.7097,
                                               'recall': 0.5197},
                                 'causal-en': {'f1': 0.6351,
                                               'precision': 0.9141,
                                               'recall': 0.4866},
                                 'causal-es': {'f1': 0.6506,
                                               'precision': 0.5867,
                                               'recall': 0.7302},
                                 'causal-tr': {'f1': 0.6346,
                                               'precision': 0.8165,
                                               'recall': 0.519},
                                 'causal-ur': {'f1': 0.6501,
             

In [34]:
from pprint import pprint
pprint(analyse_ff("../agentere/logs/run_2026-02-18T13-18-46.816746p00-00_c65c4e77.json")) # Coherence

{'clipped_sym_ec': {'overall': {'f1': 0.5956,
                                'precision': 0.4974,
                                'recall': 0.7422},
                    'per_lang': {'causal-da': {'f1': 0.5333,
                                               'precision': 0.4,
                                               'recall': 0.8},
                                 'causal-en': {'f1': 0.7805,
                                               'precision': 0.6957,
                                               'recall': 0.8889},
                                 'causal-es': {'f1': 0.4706,
                                               'precision': 0.3279,
                                               'recall': 0.8333},
                                 'causal-tr': {'f1': 0.6471,
                                               'precision': 0.6735,
                                               'recall': 0.6226},
                                 'causal-ur': {'f1': 0.6452,
               

In [35]:
from pprint import pprint
pprint(analyse_ff("../agentere/logs/run_2026-02-18T20-11-18.089432p00-00_f967d59d.json"))

{'clipped_sym_ec': {'overall': {'f1': 0.6452,
                                'precision': 0.6115,
                                'recall': 0.6828},
                    'per_lang': {'causal-da': {'f1': 0.5796,
                                               'precision': 0.5044,
                                               'recall': 0.6811},
                                 'causal-en': {'f1': 0.7098,
                                               'precision': 0.8055,
                                               'recall': 0.6344},
                                 'causal-es': {'f1': 0.5874,
                                               'precision': 0.4612,
                                               'recall': 0.8088},
                                 'causal-tr': {'f1': 0.6741,
                                               'precision': 0.6973,
                                               'recall': 0.6523},
                                 'causal-ur': {'f1': 0.608,
          

In [26]:
from pprint import pprint
keyy_ff("../old/aya/wtools_wresamp.json") 

{'/results/resampling/stats/per_doc_resampling/217_0',
 '/results/resampling/stats/aggregation_stats/per_pair_stats/T2,T25',
 '/config/paths/dataset_path',
 '/results/resampling/stats/aggregation_stats/per_pair_stats/T10,T28',
 '/results/resampling/stats/aggregation_stats/per_pair_stats/T22,T8',
 '/results/resampling/stats/aggregation_stats/per_pair_stats/T1,T0/was_tied',
 '/results/resampling/stats/per_doc_resampling/431_0',
 '/results/resampling/stats/per_doc_resampling/697_0',
 '/results/resampling/stats/per_doc_resampling/103_0',
 '/results/resampling/stats/per_doc_resampling/358_0',
 '/results/resampling/stats/aggregation_stats/per_pair_stats/T14,T1',
 '/results/resampling/stats/per_doc_resampling/208_0',
 '/results/resampling/stats/aggregation_stats/per_pair_stats/T13,T23',
 '/results/resampling/stats/aggregation_stats/tied_decisions',
 '/results/resampling/stats/aggregation_stats/per_pair_stats/T4,T5',
 '/results/resampling/stats/aggregation_stats/per_pair_stats/T30,T3',
 '/resu

In [19]:
from pprint import pprint
keyy_ff("../agentere/logs/run_2026-02-18T13-46-40.046543p00-00_51a859c5.json") 

{'/cli_args',
 '/config',
 '/config/active_dataset',
 '/config/datasets',
 '/config/datasets/event_story_line',
 '/config/datasets/maven_ere',
 '/config/datasets/maven_ere/ann_field',
 '/config/datasets/maven_ere/labels',
 '/config/datasets/maven_ere/max_examples',
 '/config/datasets/maven_ere/name',
 '/config/datasets/maven_ere/prompt',
 '/config/datasets/maven_ere/repo_id',
 '/config/datasets/maven_ere/rule_set',
 '/config/datasets/maven_ere/split',
 '/config/datasets/maven_ere/text_field',
 '/config/datasets/meci',
 '/config/encoder',
 '/config/encoder/filter_norel',
 '/config/encoder/filter_norel/delta',
 '/config/encoder/path',
 '/config/experiment',
 '/config/experiment/concurrency',
 '/config/experiment/enable_tools',
 '/config/experiment/resampling',
 '/config/experiment/resampling/enabled',
 '/config/experiment/resampling/n_runs',
 '/config/experiment/resampling/tie_breaking',
 '/config/experiment/retries',
 '/config/experiment/tools',
 '/config/experiment/tracing',
 '/config/

!python3 one_shot_scripts/reformat.py --log "logs/aid_run_2025-10-03T05-53-41.957971p00-00_9cb19f24 copy.json" --out "tools_resamp.json" --dataset Nofing/MECI-v0.1-public-span --split test

In [38]:
from pprint import pprint
# pprint(analyse_ff("../agentere/logs/run_2026-02-05T15-33-45.728643p00-00_8e9ee038.json"))
pprint(analyse_ff("../old/aya/id_run_wtools_wresamp.json"))

{'clipped_sym_ec': {'overall': {'f1': 0.7364,
                                'precision': 0.8237,
                                'recall': 0.6658},
                    'per_lang': {'causal-da': {'f1': 0.7191,
                                               'precision': 0.7824,
                                               'recall': 0.6654},
                                 'causal-en': {'f1': 0.7919,
                                               'precision': 0.9375,
                                               'recall': 0.6855},
                                 'causal-es': {'f1': 0.7301,
                                               'precision': 0.7033,
                                               'recall': 0.759},
                                 'causal-tr': {'f1': 0.7286,
                                               'precision': 0.8776,
                                               'recall': 0.6228},
                                 'causal-ur': {'f1': 0.7098,
          

In [39]:
from pprint import pprint
# pprint(analyse_ff("../agentere/logs/run_2026-02-05T15-33-45.728643p00-00_8e9ee038.json"))
pprint(analyse_ff("../old/aya/id_run_wtools_noresamp.json"))

{'clipped_sym_ec': {'overall': {'f1': 0.7352,
                                'precision': 0.7942,
                                'recall': 0.6844},
                    'per_lang': {'causal-da': {'f1': 0.7257,
                                               'precision': 0.7818,
                                               'recall': 0.6772},
                                 'causal-en': {'f1': 0.8108,
                                               'precision': 0.9184,
                                               'recall': 0.7258},
                                 'causal-es': {'f1': 0.6985,
                                               'precision': 0.6444,
                                               'recall': 0.7626},
                                 'causal-tr': {'f1': 0.7259,
                                               'precision': 0.837,
                                               'recall': 0.6408},
                                 'causal-ur': {'f1': 0.7183,
          

In [40]:
from pprint import pprint
# pprint(analyse_ff("../agentere/logs/run_2026-02-05T15-33-45.728643p00-00_8e9ee038.json"))
pprint(analyse_ff("../old/aya/id_run_notools_noresamp.json"))

{'clipped_sym_ec': {'overall': {'f1': 0.7048,
                                'precision': 0.8075,
                                'recall': 0.6253},
                    'per_lang': {'causal-da': {'f1': 0.6955,
                                               'precision': 0.7284,
                                               'recall': 0.6654},
                                 'causal-en': {'f1': 0.8185,
                                               'precision': 0.9568,
                                               'recall': 0.7151},
                                 'causal-es': {'f1': 0.6623,
                                               'precision': 0.6109,
                                               'recall': 0.723},
                                 'causal-tr': {'f1': 0.6781,
                                               'precision': 0.8864,
                                               'recall': 0.549},
                                 'causal-ur': {'f1': 0.7079,
           

In [44]:
toto = analyse_ff("../old/aya/id_run_notools_noresamp.json")

In [77]:
def and_add(things):
    res = ""
    for th in things:
        res += " & " + str(th)
    return res

def lang_str(metrics):
    return and_add([round(metrics[key]*100, 2) for key in ['precision', 'recall', 'f1']])

def meth_str(langs):
    return and_add([lang_str(langs[key]) for key in ['causal-en', 'causal-da', 'causal-es', 'causal-tr', 'causal-ur']])

def latex_str(rep):
    res = ""
    res += meth_str(rep["sym_ce"]["per_lang"]) + " \\\\" + "\n"
    res += meth_str(rep["sym_ec"]["per_lang"]) + " \\\\"
    return res

In [78]:
print(latex_str(analyse_ff("../old/aya/id_run_notools_noresamp.json")))

 &  & 71.6 & 71.51 & 71.55 &  & 55.16 & 67.32 & 60.64 &  & 44.69 & 72.66 & 55.34 &  & 78.38 & 55.1 & 64.71 &  & 62.38 & 66.67 & 64.45 \\
 &  & 84.71 & 71.51 & 77.55 &  & 64.38 & 66.54 & 65.44 &  & 50.25 & 72.3 & 59.29 &  & 84.09 & 54.9 & 66.43 &  & 66.67 & 66.67 & 66.67 \\


In [79]:
print(latex_str(analyse_ff("../old/aya/id_run_wtools_noresamp.json")))

 &  & 70.31 & 72.58 & 71.43 &  & 63.37 & 68.11 & 65.65 &  & 47.91 & 76.26 & 58.85 &  & 74.8 & 64.27 & 69.14 &  & 59.4 & 73.54 & 65.72 \\
 &  & 77.36 & 72.58 & 74.9 &  & 69.92 & 67.72 & 68.8 &  & 52.8 & 76.26 & 62.4 &  & 76.7 & 64.08 & 69.82 &  & 62.19 & 73.54 & 67.39 \\


In [80]:
print(latex_str(analyse_ff("../old/aya/id_run_wtools_wresamp.json")))

 &  & 77.51 & 68.55 & 72.75 &  & 65.89 & 66.54 & 66.21 &  & 56.72 & 75.9 & 64.92 &  & 81.82 & 62.28 & 70.73 &  & 59.57 & 72.49 & 65.39 \\
 &  & 86.15 & 68.55 & 76.35 &  & 71.31 & 66.54 & 68.84 &  & 61.34 & 75.9 & 67.85 &  & 83.1 & 62.28 & 71.2 &  & 62.99 & 72.49 & 67.4 \\


In [116]:
from pprint import pprint
# pprint(analyse_ff("../agentere/logs/run_2026-02-05T15-33-45.728643p00-00_8e9ee038.json"))
analyse_ff("../agentere/logs/run_2026-02-18T13-18-46.816746p00-00_c65c4e77.json") 

{'direct_binary': {'per_lang': {'causal-tr': {'precision': 0.5492,
    'recall': 0.6321,
    'f1': 0.5877},
   'causal-en': {'precision': 0.5161, 'recall': 0.8889, 'f1': 0.6531},
   'causal-es': {'precision': 0.2837, 'recall': 0.8333, 'f1': 0.4233},
   'causal-da': {'precision': 0.3721, 'recall': 0.8, 'f1': 0.5079},
   'causal-ur': {'precision': 0.5526, 'recall': 0.8077, 'f1': 0.6562}},
  'overall': {'precision': 0.4276, 'recall': 0.75, 'f1': 0.5447}},
 'clipped_sym_ec': {'per_lang': {'causal-tr': {'precision': 0.6735,
    'recall': 0.6226,
    'f1': 0.6471},
   'causal-en': {'precision': 0.6957, 'recall': 0.8889, 'f1': 0.7805},
   'causal-es': {'precision': 0.3279, 'recall': 0.8333, 'f1': 0.4706},
   'causal-da': {'precision': 0.4, 'recall': 0.8, 'f1': 0.5333},
   'causal-ur': {'precision': 0.5556, 'recall': 0.7692, 'f1': 0.6452}},
  'overall': {'precision': 0.4974, 'recall': 0.7422, 'f1': 0.5956}},
 'sym_ec': {'per_lang': {'causal-tr': {'precision': 0.6346,
    'recall': 0.6226,
    

In [117]:
from pprint import pprint
# pprint(analyse_ff("../agentere/logs/run_2026-02-05T15-33-45.728643p00-00_8e9ee038.json"))
analyse_ff("../agentere/logs/run_2026-02-18T13-46-40.046543p00-00_51a859c5.json") 

{'direct_binary': {'per_lang': {'causal-tr': {'precision': 0.4717,
    'recall': 0.4717,
    'f1': 0.4717},
   'causal-en': {'precision': 0.5517, 'recall': 0.8889, 'f1': 0.6809},
   'causal-es': {'precision': 0.221, 'recall': 0.8333, 'f1': 0.3493},
   'causal-da': {'precision': 0.4048, 'recall': 0.85, 'f1': 0.5484},
   'causal-ur': {'precision': 0.5405, 'recall': 0.7692, 'f1': 0.6349}},
  'overall': {'precision': 0.3777, 'recall': 0.6875, 'f1': 0.4875}},
 'clipped_sym_ec': {'per_lang': {'causal-tr': {'precision': 0.575,
    'recall': 0.434,
    'f1': 0.4946},
   'causal-en': {'precision': 0.7273, 'recall': 0.8889, 'f1': 0.8},
   'causal-es': {'precision': 0.3125, 'recall': 0.8333, 'f1': 0.4545},
   'causal-da': {'precision': 0.4474, 'recall': 0.85, 'f1': 0.5862},
   'causal-ur': {'precision': 0.6667, 'recall': 0.7692, 'f1': 0.7143}},
  'overall': {'precision': 0.4804, 'recall': 0.6719, 'f1': 0.5603}},
 'sym_ec': {'per_lang': {'causal-tr': {'precision': 0.5412,
    'recall': 0.434,
    

In [41]:
flatten_pairs_solo(df.copy())

{('epidemics-week4-ahmed-936072_chunk_0.ann', 'T2,T10', 'EffectCause'),
 ('disasters-week4-laila-140126_chunk_33.ann', 'T3,T9', 'EffectCause'),
 ('conflict-week2-ahmed-5094_chunk_18.ann', 'T4,T3', 'EffectCause'),
 ('epidemics-week2-laila-1019296_chunk_4.ann', 'T11,T3', 'EffectCause'),
 ('economic_crisis-week4-phuong-1013769_chunk_13.ann',
  'T24,T13',
  'EffectCause'),
 ('disasters-week4-isik-1741700_chunk_0.ann', 'T4,T8', 'EffectCause'),
 ('epidemics-week4-ahmed-897001_chunk_3.ann', 'T10,T5', 'EffectCause'),
 ('economic_crisis-week4-cagatay-1097085_chunk_5.ann', 'T6,T5', 'EffectCause'),
 ('aviation_accidents-week4-cagatay-2195928_chunk_17.ann',
  'T1,T11',
  'EffectCause'),
 ('epidemics-week4-ahmed-934409_chunk_0.ann', 'T9,T8', 'EffectCause'),
 ('disasters-week2-cagatay-1208810_chunk_1.ann', 'T6,T2', 'EffectCause'),
 ('economic_crisis-week4-phuong-10062100_chunk_69.ann',
  'T24,T9',
  'EffectCause'),
 ('economic_crisis-week4-isik-2112256_chunk_24.ann', 'T4,T3', 'EffectCause'),
 ('conf

In [5]:
def evaluate_event_relations(data):
    predictions = data['results']['per_pair_predictions']
    df = pd.DataFrame(predictions)

    # 1. Create a Unified Column (Binary: Causal vs NoRel)
    # If gold/pred is NOT 'NoRel', it becomes 'Causal'
    df['unified_gold'] = df['gold'].apply(lambda x: 'NoRel' if x == 'NoRel' else 'Causal')
    df['unified_pred'] = df['pred'].apply(lambda x: 'NoRel' if x == 'NoRel' else 'Causal')

    results_summary = {}

    # --- A. Global Unified Scores (Bidirectional Causal) ---
    results_summary['overall_unified'] = binary_metrics(
        df['unified_gold'], 
        df['unified_pred']
    )

    # --- B. Per Language Unified Scores ---
    results_summary['per_lang'] = {}
    languages = df['lang'].unique()
    
    for lang in languages:
        lang_df = df[df['lang'] == lang]
        results_summary['per_lang'][lang] = binary_metrics(
            lang_df['unified_gold'], 
            lang_df['unified_pred']
        )

    # --- C. Scores per Original Relation Type (One-vs-Rest Logic) ---
    # The user asked for results where only one type is used.
    # We interpret this as: How well does the model detect "EffectCause" specifically?
    # To do this, we treat the specific label as Positive and EVERYTHING ELSE (including other causal types) as Negative.
    
    # Get all unique labels except NoRel that appear in Gold or Pred
    all_labels = set(df['gold'].unique()) | set(df['pred'].unique())
    if 'NoRel' in all_labels:
        all_labels.remove('NoRel')
    
    results_summary['per_relation_type'] = {}
    
    for label in all_labels:
        # Binary mapping: 1 if it matches the current label, 0 otherwise
        # Note: Depending on your strictness, you might want to exclude other Causal types
        # from the calculation, but standard One-vs-Rest includes them as negatives.
        
        # Here we perform strict matching: Only 'EffectCause' is True, 'NoRel' OR 'CauseEffect' is False
        temp_gold = df['gold'].apply(lambda x: x if x == label else 'Other')
        temp_pred = df['pred'].apply(lambda x: x if x == label else 'Other')
        
        results_summary['per_relation_type'][label] = binary_metrics(
            temp_gold, 
            temp_pred,
            pos_label=label
        )

    return results_summary

metrics = evaluate_event_relations(input_data)

# Pretty Print Result
print(json.dumps(metrics, indent=2))

NameError: name 'input_data' is not defined

In [119]:
a = [
{"percentage": 0.01, "n_samples": 26, "micro_f1": 0.025548395156840494, "macro_f1": 0.017866592862999325, "precision": 0.014885537856822235, "recall": 0.09006165590135055, "dev_best_f1": 0.046428393895710095, "best_threshold": 0.18281687879636302, "patience_setting": 5},
{"percentage": 0.025, "n_samples": 65, "micro_f1": 0.0347387304589118, "macro_f1": 0.017926586204941693, "precision": 0.018481950595691462, "recall": 0.28853493834409866, "dev_best_f1": 0.030428224801626335, "best_threshold": 0.2588339133269339, "patience_setting": 5},
{"percentage": 0.05, "n_samples": 131, "micro_f1": 0.16834291736351348, "macro_f1": 0.18252172123778573, "precision": 0.1223646960865945, "recall": 0.2696711685261304, "dev_best_f1": 0.17748287209365055, "best_threshold": 0.09729771494947076, "patience_setting": 5},
{"percentage": 0.1, "n_samples": 262, "micro_f1": 0.24043352505069981, "macro_f1": 0.27807880559928144, "precision": 0.2197508356122759, "recall": 0.26541397533763944, "dev_best_f1": 0.268176778076984, "best_threshold": 0.17331474948004164, "patience_setting": 5},
{"percentage": 0.25, "n_samples": 655, "micro_f1": 0.2575685817333965, "macro_f1": 0.2815229771022271, "precision": 0.25569620253164554, "recall": 0.25946858485026425, "dev_best_f1": 0.3138445523941707, "best_threshold": 0.23032752537796983, "patience_setting": 5},
{"percentage": 0.5, "n_samples": 1310, "micro_f1": 0.2754540041679071, "macro_f1": 0.296860742177667, "precision": 0.279362922705314, "recall": 0.2716529653552554, "dev_best_f1": 0.3423762198943504, "best_threshold": 0.23982965469429118, "patience_setting": 5},
{"percentage": 1.0, "n_samples": 2621, "micro_f1": 0.29256594724220625, "macro_f1": 0.3112650163801458, "precision": 0.26479157994886127, "recall": 0.32684967704051676, "dev_best_f1": 0.34877036090705843, "best_threshold": 0.16381262016372028, "patience_setting": 5},
]

In [121]:
l = [[a[i][k] for i in range(len(a))] for k in a[0].keys()]

In [135]:
for i in a[0].keys(): print(i, end=", ")
print("")
for i in a:
    for j in i.values():
        print(j, end=", ")
    print("")

percentage, n_samples, micro_f1, macro_f1, precision, recall, dev_best_f1, best_threshold, patience_setting, 
0.01, 26, 0.025548395156840494, 0.017866592862999325, 0.014885537856822235, 0.09006165590135055, 0.046428393895710095, 0.18281687879636302, 5, 
0.025, 65, 0.0347387304589118, 0.017926586204941693, 0.018481950595691462, 0.28853493834409866, 0.030428224801626335, 0.2588339133269339, 5, 
0.05, 131, 0.16834291736351348, 0.18252172123778573, 0.1223646960865945, 0.2696711685261304, 0.17748287209365055, 0.09729771494947076, 5, 
0.1, 262, 0.24043352505069981, 0.27807880559928144, 0.2197508356122759, 0.26541397533763944, 0.268176778076984, 0.17331474948004164, 5, 
0.25, 655, 0.2575685817333965, 0.2815229771022271, 0.25569620253164554, 0.25946858485026425, 0.3138445523941707, 0.23032752537796983, 5, 
0.5, 1310, 0.2754540041679071, 0.296860742177667, 0.279362922705314, 0.2716529653552554, 0.3423762198943504, 0.23982965469429118, 5, 
1.0, 2621, 0.29256594724220625, 0.3112650163801458, 0.26